# # 06 · Clustering and label transfer

Companion to the tutorial notebook [`06_clustering_transfer`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/tree/main/notebooks/level2).

**Kernel:** `cajal-lipidomics` · **Reads:** `05_embedded.h5ad` · **Writes:** `data/derived/06_clustered.h5ad`

We turn the Harmonized lipid embedding into **lipizones**, data-driven territories of lipid composition, and give every pixel a name. Four movements:

1. **Cluster** the control brain: a kNN graph, cut into communities by Leiden
2. **Compare to anatomy**: how much do lipizones agree with the Allen regions?
3. **Transfer labels** onto the pregnant brain by a kNN vote in the shared space
4. **Run EUCLID**, the package behind the atlas, and compare its divisive lipizones with ours

Our lipizones won't match the paper's exactly (two sections, not a whole atlas). The broad biology survives anyway.

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------
from lipid_project import paths
paths.ensure_dirs()
paths.summary()

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier

import cajal_lipidomics as cl
from cajal_lipidomics import analysis, embedding, plotting
from cajal_lipidomics.style import FS
cl.style.set_style()

SEED = 0
np.random.seed(SEED)    # some libraries (EUCLID) use numpy's global random state
N_JOBS = os.cpu_count() or 1    # use every CPU core for neighbour searches
print("ready. cajal_lipidomics", cl.__version__)

# ## 1. Load the embedded sections

We cluster on `X_harmony`: the NMF programs with the section offset removed. Clustering a dozen de-noised, de-batched numbers is better and much faster than clustering ~100 raw lipid columns.

In [ ]:
adata = ad.read_h5ad(paths.stage("embedded"))    # notebook 05's output
G_nmf = adata.obsm["X_nmf"].astype(float)          # lipid programs per pixel
G = adata.obsm["X_harmony"].astype(float)          # same, Harmony-corrected: what we cluster and transfer on
is_ctrl = (adata.obs["Condition"] == "naive").to_numpy()    # True = control pixel
is_preg = ~is_ctrl    # the pregnant pixels
z, y = adata.obs["zccf"].to_numpy(), -adata.obs["yccf"].to_numpy()     # plotting coordinates, dorsal up
print("pixels:", adata.n_obs, "| embedding:", G.shape, f"| control {is_ctrl.sum()} / pregnant {is_preg.sum()}")

# A first look: four programs painted on the control section
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for k, ax in enumerate(axes[: G_nmf.shape[1]]):    # one panel per program (first four)
    v = G_nmf[is_ctrl, k]    # program k's activity on control pixels
    ax.scatter(z[is_ctrl], y[is_ctrl], c=v, cmap="plasma", s=3, rasterized=True,
               vmin=np.quantile(v, 0.02), vmax=np.quantile(v, 0.98))
    cl.style.spatial_axes(ax)
    ax.set_title(f"NMF program {k}", fontsize=FS["m"])
plt.tight_layout()
plt.show()

# ## 2. The kNN graph: write down who is near whom

For each pixel, find its **k nearest neighbours** in the 12-dimensional embedding (ordinary Euclidean distance) and draw an edge to each. Pixels with similar lipid composition get wired together, even if they sit on opposite sides of the brain: we group by lipid identity, not location.

On a small sample first, so you can see the graph is just a table of edges.

In [ ]:
rng = np.random.default_rng(SEED)
sample = rng.choice(np.where(is_ctrl)[0], size=min(2000, is_ctrl.sum()), replace=False)    # 2000 random control pixels (drawing all edges would be unreadable)
Gs = G[sample]    # their harmonized coordinates
k = 15    # neighbours per pixel
dist, nb = NearestNeighbors(n_neighbors=k + 1).fit(Gs).kneighbors(Gs)    # distances and indices of the k+1 nearest
dist, nb = dist[:, 1:], nb[:, 1:]                       # column 0 is each point itself; drop it
print(f"{len(Gs)} pixels x {k} neighbours = {len(Gs) * k} directed edges")
print("pixel 0's first neighbours:", nb[0][:5], "at distances", dist[0][:5].round(3))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for i in range(len(Gs)):    # one faint line per edge
    for jj in nb[i]:
        ax.plot([Gs[i, 0], Gs[jj, 0]], [Gs[i, 1], Gs[jj, 1]], lw=0.1, color="0.6", alpha=0.3, rasterized=True)
ax.scatter(Gs[:, 0], Gs[:, 1], s=6, c="crimson", zorder=3, rasterized=True)
ax.set_xlabel("Harmonized program 0"); ax.set_ylabel("Harmonized program 1")
ax.set_title(f"kNN graph on {len(Gs)} control pixels (drawn in 2 of the dimensions)", fontsize=FS["m"])
plt.tight_layout()
plt.show()

# We only drew 2 of the 12 dimensions, so two pixels can look far apart here yet be neighbours through the other 10. Trust the graph, use the picture as intuition.

## 3. Leiden: cut the graph into communities

**Leiden** finds groups of pixels densely wired to each other and sparsely wired to the rest, like friend groups in a social network. It maximises **modularity**: the fraction of edges inside communities minus what you'd expect if edges were thrown down at random. It never sees coordinates or anatomy, and it picks the number of clusters itself.

The atlas recipe: a k = 40 graph, then `leidenalg` with the modularity partition. We build it by hand on **all** control pixels.

In [ ]:
import igraph as ig
import leidenalg


def leiden_by_hand(emb, n_neighbors=40, resolution=None, seed=230598):
    """kNN graph on `emb` -> igraph -> Leiden. Plain modularity, or resolution-tunable if given."""
    # sparse adjacency: knn[i, j] = 1 if j is one of i's neighbours
    knn = NearestNeighbors(n_neighbors=n_neighbors, n_jobs=N_JOBS).fit(emb).kneighbors_graph(emb)
    src, dst = knn.nonzero()                                   # one (source, target) pair per edge
    g = ig.Graph(n=emb.shape[0], edges=list(zip(src.tolist(), dst.tolist())))    # the same graph as an igraph object
    g.simplify()                                               # drop duplicate edges and self-loops
    if resolution is None:    # plain modularity: Leiden chooses the granularity
        part = leidenalg.find_partition(g, leidenalg.ModularityVertexPartition, n_iterations=5, seed=seed)
    else:    # resolution-tunable: higher = more, smaller clusters
        part = leidenalg.find_partition(g, leidenalg.RBConfigurationVertexPartition,
                                        resolution_parameter=resolution, n_iterations=5, seed=seed)
    return np.array(part.membership)                           # one integer cluster id per pixel


leiden_ctrl = leiden_by_hand(G[is_ctrl])    # cluster ALL control pixels
n_clusters = len(np.unique(leiden_ctrl))    # how many lipizones it found
print(f"Leiden found {n_clusters} lipizones on the control section")
print(pd.Series(leiden_ctrl).value_counts().sort_index().to_string())

# The tutorial helper should be the same recipe: check on a subset (faster)
sub = np.sort(rng.choice(is_ctrl.sum(), size=min(6000, is_ctrl.sum()), replace=False))    # a random subset of control pixels for the comparison
# identical labels from ours and the helper?
same = (leiden_by_hand(G[is_ctrl][sub]) == embedding.leiden_clusters(G[is_ctrl][sub])).all()
print("\nhand-built Leiden matches embedding.leiden_clusters on a subset:", bool(same))

# **Resolution** is the granularity dial: higher resolution prefers more, smaller, tighter clusters. The atlas doesn't reach its hundreds of lipizones by cranking this up; it refines coarse clusters with a divisive splitter (section 6). There's no single correct number: it's a scientific judgement about how fine a territory is still meaningful.

In [ ]:
for res in [0.5, 2.0]:    # a low and a high resolution
    lab = leiden_by_hand(G[is_ctrl], n_neighbors=15, resolution=res, seed=SEED)
    print(f"resolution {res:>4}  ->  {len(np.unique(lab)):>3} clusters")

plotting.cluster_size_hist(leiden_ctrl)        # sanity check: a spread of sizes, no single giant blob
plt.show()

# ## 4. Lipid territories in space, and how they track anatomy

`lipizone_colors` groups lipizones into colour families by how similar their lipid centroids are, so molecularly similar lipizones get similar colours.

In [ ]:
adata.obs["lipizone"] = "NA"    # start everyone as "NA"; control pixels get their cluster now, pregnant later
adata.obs.loc[is_ctrl, "lipizone"] = leiden_ctrl.astype(str)    # labels as strings (they're names, not quantities)
color_map = plotting.lipizone_colors(adata[is_ctrl].copy(), key="lipizone", rep="X_nmf")   # {label: hex}
adata.obs["lipizone_color"] = adata.obs["lipizone"].map(color_map).fillna("#dddddd")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
axes[0].scatter(z[is_ctrl], y[is_ctrl], c=adata.obs.loc[is_ctrl, "lipizone_color"], s=4, rasterized=True)
axes[0].set_title(f"Leiden lipizones from lipids ({n_clusters})", fontsize=FS["m"])
axes[1].scatter(z[is_ctrl], y[is_ctrl], c=adata.obs.loc[is_ctrl, "allencolor"], s=4, rasterized=True)
axes[1].set_title("Allen regions (classical anatomy)", fontsize=FS["m"])
for ax in axes:
    cl.style.spatial_axes(ax)
plt.tight_layout()
plt.show()

# Spatially coherent territories, found from lipids alone. The shapes rhyme with anatomy but don't match exactly. One number for agreement: the **Adjusted Rand Index** (ARI), which is 1 for identical partitions and ~0 for unrelated ones, corrected for chance.

In [ ]:
# agreement between two labelings, corrected for chance
ari = adjusted_rand_score(adata.obs.loc[is_ctrl, "acronym"].astype(str), leiden_ctrl)
print(f"ARI(lipizones, Allen regions) on the control section = {ari:.3f}")

# A modest ARI is the lesson: lipizones partly track anatomy but are their own organising principle (one region can split into several lipizones; several regions can share one).

### Reciprocal enrichment: which lipizone lives in which region?

A plain crosstab of counts is dominated by big regions. **Reciprocal enrichment** (the atlas's metric) asks two questions and multiplies them: how enriched is this lipizone *inside* this region, and how enriched is this region *inside* this lipizone? A pair scores high only if each is over-represented in the other. Toy first.

In [ ]:
def reciprocal_enrichment(rows, cols):
    """rows x cols matrix: (col enriched in row) * (row enriched in col). >1 means mutual over-representation."""
    c1 = pd.crosstab(rows, cols)                    # counts, rows x cols
    n1 = c1 / c1.sum()                              # each column -> its distribution over rows
    n1 = (n1.T / n1.T.mean()).T                     # divide each row by its mean -> enrichment of col within row
    c2 = pd.crosstab(cols, rows)    # the other direction: row categories within column categories
    n2 = c2 / c2.sum()    # each column of c2 -> fractions
    n2 = (n2.T / n2.T.mean()).T                     # enrichment of row within col
    return n1 * n2.T    # multiply: high only if each is enriched in the other


toy_clu = pd.Series(list("AAAABBBBAB"), name="cluster")    # 10 toy pixels: their cluster...
toy_reg = pd.Series(list("XXXXYYYYYX"), name="region")    # ...and their region
print(reciprocal_enrichment(toy_reg, toy_clu).round(2))

# A scores high in X and B in Y. Now the real control section (regions with at least 0.5% of the pixels, so averages mean something), ordered so dark cells fall near the diagonal, displayed raw with colour limits at the 2nd/98th percentile.

In [ ]:
from scipy.cluster import hierarchy as sch
import seaborn as sns

reg_c = pd.Series(adata.obs.loc[is_ctrl, "acronym"].astype(str).to_numpy(), name="region")    # Allen region per control pixel
lip_c = pd.Series(leiden_ctrl, name="lipizone")    # lipizone per control pixel
big = reg_c.value_counts()    # pixels per region
keep = reg_c.isin(big.index[big >= 0.005 * len(reg_c)]).to_numpy()    # regions with >= 0.5% of the pixels
enr = reciprocal_enrichment(reg_c[keep], lip_c[keep])    # regions x lipizones

# Columns ordered by hierarchical clustering (optimal leaf order); rows by where their maximum sits
# order lipizones so similar ones sit side by side
col_order = sch.leaves_list(sch.linkage(sch.distance.pdist(enr.T), method="weighted", optimal_ordering=True))
enr = enr.iloc[:, col_order]
enr = enr.iloc[np.argsort(np.argmax(enr.values, axis=1)), :]    # order regions by their best lipizone -> near-diagonal

plt.figure(figsize=(10, 7))
sns.heatmap(enr, cmap="Purples", yticklabels=True, cbar_kws={"label": "reciprocal enrichment"},
            vmin=np.percentile(enr.values, 2), vmax=np.percentile(enr.values, 98))
plt.xlabel("lipizone"); plt.ylabel("Allen region"); plt.title("reciprocal enrichment: region x lipizone")
plt.tight_layout()
plt.show()

# ### Naming lipizones by their marker lipids

A lipizone means little until we can say what it's made of. **Marker lipids**: for each lipizone, log2(mean inside / mean outside) per lipid, top few. We use the real uMAIA intensities, labelled by lipid name.

In [ ]:
ctrl_view = ad.AnnData(np.asarray(adata.layers["umaia"])[is_ctrl], obs=adata.obs[is_ctrl].copy())    # control pixels, real uMAIA intensities
ctrl_view.var_names = ad.utils.make_index_unique(pd.Index(adata.var["lipid"].astype(str)))    # columns labelled by lipid name
ctrl_view.obs["leiden"] = pd.Categorical(leiden_ctrl.astype(str))    # the grouping the helper will use
markers = analysis.marker_lipids(ctrl_view, "leiden", top_n=4)       # {lipizone: [(lipid, log2FC), ...]}

sizes = pd.Series(leiden_ctrl.astype(str)).value_counts()    # lipizones sorted by size
for lz in sizes.index[:4]:
    print(f"lipizone {lz:>3} ({sizes[lz]:>6} px): " + ", ".join(f"{n} ({f:+.2f})" for n, f in markers[lz]))
myelin_like = [lz for lz, top in markers.items() if top and top[0][0].startswith(("HexCer", "SM", "Cer"))]    # lipizones whose top marker is a sphingolipid
print("\nsphingolipid-headed (white-matter-like) lipizones:", sorted(myelin_like, key=int))

# The full fingerprint at once: a lipizone x lipid heatmap (relative abundance), both axes cosine-clustered.

In [ ]:
plotting.sorted_lipid_heatmap(ctrl_view, group_key="leiden", figsize=(14, 6), title="lipizone x lipid (sorted)")
plt.show()

# ## 5. Label transfer: carry control lipizones onto the pregnant brain

We don't re-cluster the pregnant brain and then puzzle over which new cluster matches which. We **carry the labels over**: each pregnant pixel looks at its 15 nearest control pixels in the shared Harmony space and takes a (distance-weighted) **majority vote**. How unanimous the vote is gives a free **confidence**.

This only works because Harmony put both sections in one coordinate system (harmonized on *section*, so pregnancy biology isn't the thing being removed). In the un-harmonized space, pregnant pixels would mostly find other pregnant-looking neighbours.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=15, weights="distance")      # closer neighbours get a bigger vote
knn.fit(G[is_ctrl], leiden_ctrl.astype(str))    # learn from labelled control pixels
pred_preg = knn.predict(G[is_preg])    # the winning label for each pregnant pixel
conf_preg = knn.predict_proba(G[is_preg]).max(axis=1)                # share of the vote the winner got

print(f"transferred {len(pred_preg):,} labels | mean confidence {conf_preg.mean():.1%} | "
      f"above 0.8: {(conf_preg > 0.8).mean():.1%}")
pred_h, _ = embedding.knn_transfer(G[is_ctrl], leiden_ctrl.astype(str), G[is_preg], k=15)    # the helper version, for a check
print("helper agrees on", f"{(pred_h == pred_preg).mean():.1%}", "of pregnant pixels")

adata.obs.loc[is_preg, "lipizone"] = pred_preg    # pregnant pixels now carry transferred lipizones
adata.obs["lipizone_color"] = adata.obs["lipizone"].map(color_map)    # same colour = same lipizone, in both sections
adata.obs["transfer_confidence"] = np.nan    # control pixels have no transfer confidence
adata.obs.loc[is_preg, "transfer_confidence"] = conf_preg

fig, axes = plt.subplots(1, 3, figsize=(17, 5.4))
axes[0].scatter(z[is_ctrl], y[is_ctrl], c=adata.obs.loc[is_ctrl, "lipizone_color"], s=4, rasterized=True)
axes[0].set_title("control: lipizones learned with Leiden", fontsize=FS["m"])
axes[1].scatter(z[is_preg], y[is_preg], c=adata.obs.loc[is_preg, "lipizone_color"], s=4, rasterized=True)
axes[1].set_title("pregnant: same lipizones, transferred", fontsize=FS["m"])
sc = axes[2].scatter(z[is_preg], y[is_preg], c=conf_preg, cmap="viridis", vmin=0.3, vmax=1, s=4, rasterized=True)
axes[2].set_title("transfer confidence (low = boundaries)", fontsize=FS["m"])
for ax in axes:
    cl.style.spatial_axes(ax)
cl.style.lightweight_colorbar(sc, axes[2], label="vote share")
plt.tight_layout()
plt.show()

# Same colours mean the same lipizone in both brains, so a territory changing size or position is a candidate biological change. Low-confidence pixels cluster at boundaries, the ones a curator would review.

### Reading the lipizones a few more ways

- **The hierarchy:** group lipizones by centroid similarity and cut at 2, 4, 8, 16 groups, to see which coarse division each fine lipizone belongs to.
- **One lipizone in red** over a grey cast of everything else.
- **A zoomed mosaic** of the dorsal third (cortex + hippocampus).
- **The t-SNE** coloured by lipizone and by Allen region.

In [ ]:
ctrl_ad = adata[is_ctrl].copy()    # the control section alone
levels = plotting.lipizone_hierarchy_levels(ctrl_ad, key="lipizone", rep="X_nmf", max_level=4)    # each pixel's group at 2, 4, 8, 16 groups
plotting.plot_hierarchy_levels(ctrl_ad, levels, color_key="lipizone_color")
plt.suptitle("the lipizone hierarchy, split by split", y=1.02)
plt.show()

biggest = adata.obs.loc[is_ctrl, "lipizone"].value_counts().index[0]    # the largest lipizone
plotting.highlight_lipizone(adata, biggest, key="lipizone")
plt.suptitle(f"lipizone {biggest} (the largest) in red", y=1.02)
plt.show()

In [ ]:
xs, ys = adata.obs.loc[is_ctrl, "x"], adata.obs.loc[is_ctrl, "y"]    # pixel coordinates of the control section
box_x = (xs.quantile(0.25), xs.quantile(0.75))      # edit these to roam around the section
box_y = (ys.quantile(0.05), ys.quantile(0.45))    # the dorsal part (small y = top of the image)
ctrl_section = adata.obs.loc[is_ctrl, "SectionID"].iloc[0]    # the control section's id
plotting.mosaic(adata, section=ctrl_section, xlim=box_x, ylim=box_y, point_size=22)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
plotting.tsne_colored(adata, by="lipizone_color", kind="color", ax=axes[0], title="t-SNE by lipizone")
plotting.tsne_colored(adata, by="allencolor", kind="color", ax=axes[1], title="t-SNE by Allen region")
plt.tight_layout()
plt.show()

# ## 6. The production engine: EUCLID

The atlas uses **EUCLID** (Enhanced uMAIA for Clustering Lipizones, Imputation, and Differential analysis). Instead of one flat Leiden, it's a **divisive, top-down binary splitter**. At every node it:
1. recomputes a *local* NMF on just that branch's pixels, so fine structure surfaces
2. over-segments with k-means into K micro-clusters, then re-aggregates them into two groups (**backSPIN**, a biclustering routine that sorts micro-clusters along a gradient and finds the cleanest cut)
3. accepts the split only if it passes gates: enough lipids differ (Mann-Whitney + BH), both halves are big enough, both are spatially coherent
4. trains an **XGBoost classifier** for that split, which is also how new pixels get assigned later

`setup.sh` cloned EUCLID into `external/EUCLID`. Its package `__init__` imports everything and downloads reference data, so we **load only the clustering module** by registering an empty package shell first. We also stub out `squidpy` (imported at the top of the module, never used on this path, and heavy to install), and bridge one small xgboost API change.

In [ ]:
import sys, types, importlib, warnings

EUCLID_SRC = paths.REPO_ROOT / "external" / "EUCLID" / "src" / "euclid_msi"
# fail early with a clear message if setup.sh hasn't been run
assert EUCLID_SRC.is_dir(), "EUCLID not found: run `bash scripts/setup.sh` (it clones it into external/)"

# 1) An empty "euclid_msi" package whose __path__ points at the real folder. Python can then load
#    euclid_msi.clustering (and its relative import of backSPIN) WITHOUT running the real __init__.py.
pkg = types.ModuleType("euclid_msi")
pkg.__path__ = [str(EUCLID_SRC)]
sys.modules["euclid_msi"] = pkg

# 2) clustering.py does `import squidpy as sq` at the top but our path never calls it.
#    A stub module satisfies the import and fails loudly only if something actually uses it.
if "squidpy" not in sys.modules:
    sq = types.ModuleType("squidpy")
    def _no_squidpy(*a, **k):
        raise RuntimeError("squidpy is intentionally not installed; this code path doesn't use it")
    class _NoSquidpy:                        # any attribute you ask for (sq.gr.whatever) is the raiser
        def __getattr__(self, name):
            return _no_squidpy
    sq.gr = _NoSquidpy()
    sys.modules["squidpy"] = sq

euc = importlib.import_module("euclid_msi.clustering")    # loads clustering.py (and backSPIN) only
warnings.filterwarnings("ignore")      # EUCLID itself silences warnings globally; keep the output readable

# 3) Newer xgboost moved `early_stopping_rounds` from .fit() to the constructor. Bridge EUCLID's older call.
import xgboost as xgb
class _XGB(xgb.XGBClassifier):
    def fit(self, X, y, eval_set=None, early_stopping_rounds=None, verbose=False, **kw):
        if early_stopping_rounds is not None and eval_set is not None:
            self.set_params(early_stopping_rounds=early_stopping_rounds)
        return super().fit(X, y, eval_set=eval_set, verbose=verbose, **kw)
euc.XGBClassifier = _XGB    # EUCLID now builds our bridged classifier
print("EUCLID clustering module loaded from", EUCLID_SRC)

# EUCLID's `Clustering` object expects an embedding bundle: the lipid matrix (local NMF runs on it), the standardized global embedding, the t-SNE, and per-pixel coordinates with a section column.

Its spatial **continuity gate** was built for a *stack* of serial sections along the anterior-posterior axis. We have a single coronal plane, so that check can't apply. We mark all pixels as one section (EUCLID then uses a plain random 60/20/20 train/val/test split) and bypass the continuity check.

In [ ]:
from sklearn.preprocessing import StandardScaler

ctrl_full = adata[is_ctrl]    # the control section
lipid_names = ad.utils.make_index_unique(pd.Index(ctrl_full.var["lipid"].astype(str)))
X_lip = np.asarray(ctrl_full.layers["umaia"], dtype=np.float64)    # uMAIA lipids, as EUCLID expects (float64)
X_approx = X_lip + np.abs(rng.normal(0, 1e-6, X_lip.shape))    # tiny positive jitter: no perfectly flat column, still >= 0

# replace the stack-continuity check with one that always passes (single section)
euc.Clustering._continuity_check = lambda self, spat, kmeans_labels, **kw: (True, True, 1, 1, 1e9, 1e9)

bundle = types.SimpleNamespace()    # a plain object to hang `.adata` on, like EUCLID's embedding object
ae = ad.AnnData(X=X_lip.copy())    # the AnnData EUCLID reads
ae.var_names, ae.obs_names = lipid_names, ctrl_full.obs_names
ae.obs["x"] = ctrl_full.obs["zccf"].to_numpy()    # EUCLID's spatial x = our left-right CCF axis
ae.obs["y"] = ctrl_full.obs["yccf"].to_numpy()    # EUCLID's spatial y = our dorsal-ventral CCF axis
ae.obs["SectionID"] = 0                                          # one real coronal section
ae.obsm["X_Harmonized"] = ctrl_full.obsm["X_harmony"].astype(np.float64)    # the global embedding (context at every node)
ae.obsm["X_approximated"] = X_approx    # the matrix local NMF runs on
ae.obsm["X_TSNE"] = ctrl_full.obsm["X_tsne"].astype(np.float64)    # only used for EUCLID's optional plots
bundle.adata = ae

euclid_dir = paths.DERIVED / "euclid"                            # EUCLID writes its tree + log here
euclid_dir.mkdir(parents=True, exist_ok=True)    # data/derived/euclid/ (git-ignored)
clust = euc.Clustering(bundle, analysis_name=str(euclid_dir / "control"))    # files will be named control_*

# The differential gate runs Mann-Whitney on a percentile-clipped lipid table; flat lipids can become
# NaN there, so replace NaN with 0 and add a whisper of jitter so ties don't break the test.
L = clust.adatamaia.obsm["lipids"]    # the lipid table EUCLID tests splits on
clust.adatamaia.obsm["lipids"] = pd.DataFrame(np.nan_to_num(L.values) + rng.normal(0, 1e-6, L.shape),
                                              index=L.index, columns=L.columns)
print("EUCLID input ready:", ctrl_full.n_obs, "control pixels x", len(lipid_names), "lipids")

# **Learn the tree** on every control pixel. With `max_depth=7` this can resolve up to 2⁷ = 128 leaves (the gates prune some). It takes a few minutes. Each leaf's barcode is the string of 1/2 decisions taken down the tree.

In [ ]:
np.random.seed(SEED)                         # EUCLID shuffles its train/val/test split with np.random
root_node, clustering_log = clust.learn_euclid_clustering(
    K=15,                 # over-segment each branch into 15 micro-clusters before re-aggregating to 2
    min_voxels=100,       # don't split a group smaller than 100 pixels
    min_diff_lipids=2,    # a split needs more than 2 differential lipids
    min_fc=0.2, pthr=0.05,
    ACCTHR=0.6,           # the node's XGBoost must reach 60% test accuracy, or the split is dropped
    max_depth=7,
    combinations=2,       # how many top-factor combinations to try per split
    xgb_n_estimators=40, xgb_n_jobs=N_JOBS,
    do_plotting=False,
)
lvl = [c for c in clustering_log.columns if c.startswith("level_")]    # one column per tree level
# glue each pixel's per-level 1/2 decisions into a barcode string
euclid_leaf = clustering_log[lvl].fillna(0).astype(int).astype(str).agg("".join, axis=1)
print("EUCLID learned", euclid_leaf.nunique(), "lipizones on the control brain")

# **Apply the tree** to both sections: every pixel walks from the root to a leaf, the XGBoost classifier at each node deciding left or right. That's EUCLID's label transfer, the divisive cousin of our kNN vote. (EUCLID prints a leftover debug line, `ciao`, at the end of this step. Harmless.)

In [ ]:
full_ad = ad.AnnData(X=np.asarray(adata.layers["umaia"], dtype=np.float64))    # both sections, same lipids
full_ad.var_names = ad.utils.make_index_unique(pd.Index(adata.var["lipid"].astype(str)))
full_ad.obs_names = adata.obs_names    # keep pixel ids so results line up
global_full = pd.DataFrame(StandardScaler().fit_transform(adata.obsm["X_harmony"]), index=adata.obs_names)    # the standardized global embedding for every pixel

paths_df = clust.apply_euclid_clustering(root_node, full_ad, global_full)    # walk every pixel down the tree
cols = [c for c in paths_df.columns if c.startswith("level_")]    # the per-level decisions
adata.obs["euclid_lipizone"] = (paths_df[cols].fillna(0).astype(int).astype(str)
                                .agg("".join, axis=1).reindex(adata.obs_names).values)
print("EUCLID lipizones on both sections:", adata.obs["euclid_lipizone"].nunique())
print(adata.obs.groupby("Condition")["euclid_lipizone"].nunique().to_string())

In [ ]:
euclid_cmap = plotting.lipizone_colors(adata[is_ctrl].copy(), key="euclid_lipizone", rep="X_nmf")    # a similarity-ordered palette for EUCLID's lipizones
fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
axes[0].scatter(z[is_ctrl], y[is_ctrl], c=adata.obs.loc[is_ctrl, "lipizone_color"], s=4, rasterized=True)
axes[0].set_title(f"our Leiden lipizones ({n_clusters})", fontsize=FS["m"])
axes[1].scatter(z[is_ctrl], y[is_ctrl], c=adata.obs.loc[is_ctrl, "euclid_lipizone"].map(euclid_cmap),
                s=4, rasterized=True)
axes[1].set_title(f"EUCLID lipizones ({adata.obs['euclid_lipizone'].nunique()}, divisive tree)", fontsize=FS["m"])
for ax in axes:
    cl.style.spatial_axes(ax)
plt.tight_layout()
plt.show()
ari_le = adjusted_rand_score(adata.obs.loc[is_ctrl, "lipizone"], adata.obs.loc[is_ctrl, "euclid_lipizone"])    # how similar are the two clusterings?
print(f"ARI(our Leiden, EUCLID) on control = {ari_le:.3f}")

# The big divisions (cortex, white matter, deep nuclei) appear in both, EUCLID resolving them more finely.

## 7. Save

We carry forward the lipizones we can fully account for (Leiden on control + kNN transfer), keeping EUCLID's labels alongside for comparison.

In [ ]:
adata.obs["lipizone"] = pd.Categorical(adata.obs["lipizone"].astype(str))    # categorical: compact, and order-able
for col in adata.obs.columns:                          # h5ad wants consistent types: object columns -> plain strings
    if adata.obs[col].dtype == object:
        adata.obs[col] = adata.obs[col].astype(str)
adata.uns["lipizone_colors"] = color_map               # so later notebooks reuse the same palette
adata.write_h5ad(paths.stage("clustered"))             # -> data/derived/06_clustered.h5ad
print("saved", paths.stage("clustered"))
print("lipizones per condition:", adata.obs.groupby("Condition", observed=True)["lipizone"].nunique().to_dict())

# ## Recap

- A **kNN graph** is just a table of who-is-near-whom; **Leiden** cuts it into communities (lipizones), **resolution** sets granularity.
- Lipizones are spatially coherent and **partly** track Allen anatomy (modest ARI), which reciprocal enrichment and marker lipids make concrete.
- **kNN label transfer** in the Harmony space puts the pregnant brain in the same vocabulary, with a confidence map.
- **EUCLID**'s divisive tree + per-node XGBoost is the careful, production version of the same learn-then-transfer logic.
- Saved **`data/derived/06_clustered.h5ad`** with `obs["lipizone"]`, `obs["euclid_lipizone"]`, `obs["transfer_confidence"]`.

**Next (07):** build the clustering from scratch, then find what pregnancy changes.